In [ ]:
import openpyxl
import pandas as pd
import os

# ── CONFIGURATION ──────────────────────────────────────────────────────────

FOLDER_PATH = r""

# ── LOOP THROUGH ALL EXCEL FILES ───────────────────────────────────────────

all_colored_rows = []

for filename in os.listdir(FOLDER_PATH):
    if filename.endswith('.xlsx'):
        file_path = os.path.join(FOLDER_PATH, filename)
        print(f"Processing: {filename}")

        wb     = openpyxl.load_workbook(file_path, data_only=True)
        sheet  = wb.active
        header = [cell.value for cell in sheet[1]]

        for row in sheet.iter_rows(min_row=2):
            first_cell_fill = row[0].fill.start_color.index

            if first_cell_fill and first_cell_fill not in ['00000000', '00FFFFFF', 0]:
                row_data = {col: cell.value for col, cell in zip(header, row)}
                row_data['source_file'] = filename
                all_colored_rows.append(row_data)

        wb.close()
        print(f"  → {len(all_colored_rows)} colored rows so far")

# ── COMBINE ALL INTO ONE DATAFRAME ─────────────────────────────────────────

final_df = pd.DataFrame(all_colored_rows)

# ── DROP FLAG COLUMNS ──────────────────────────────────────────────────────

flag_columns = [
    'flag_zero_price', 'flag_unit_price_outlier', 'qty_range_outlier',
    'flag_qty_outlier', 'flag_duplicate', 'flag_round_numbers',
    'flag_fraction_qty', 'flag_implausible_qty', 'flag_missing_values'
]

final_df = final_df.drop(columns=[col for col in flag_columns if col in final_df.columns])

# ── SAVE ───────────────────────────────────────────────────────────────────

final_df.to_excel('all_colored_errors_c.xlsx', index=False)
print(f"\nDone. Total colored rows found: {len(final_df)}")

Processing: белгиланган_гр_маҳсулотлар_C.xlsx
  → 2 colored rows so far
Processing: белгиланган_дона_маҳсулотлар_C.xlsx
  → 23 colored rows so far
Processing: белгиланган_ижара_маҳсулотлар_C.xlsx
  → 23 colored rows so far
Processing: белгиланган_квм_маҳсулотлар_C.xlsx
  → 23 colored rows so far
Processing: белгиланган_кун_маҳсулотлар_C.xlsx
  → 23 colored rows so far
Processing: белгиланган_курс_маҳсулотлар_C.xlsx
  → 23 colored rows so far
Processing: белгиланган_литр_маҳсулотлар_C.xlsx
  → 25 colored rows so far
Processing: белгиланган_метр_маҳсулотлар_C.xlsx
  → 25 colored rows so far
Processing: белгиланган_тўплам_маҳсулотлар_C.xlsx
  → 25 colored rows so far
Processing: белгиланган_хизмат_маҳсулотлар_C.xlsx
  → 25 colored rows so far

Done. Total colored rows found: 25


In [21]:
import pandas as pd

# ── CONFIGURATION ──────────────────────────────────────────────────────────

ORIGINAL_PATH  = r"C:\Users\s.bektemirov\Desktop\Python_VS\Уй_хўжалиги_сўровномаси_нарх1_2026_-_all_versions_-_Uzbek_uz_-_2026-06-19-04-13-32.xlsx"   # paste original file path here
OUTPUT_FOLDER  = r"C:\Users\s.bektemirov\Desktop\Python_VS\foodselc_\Flagged_city"   # paste folder path where city files will be saved

ORIGINAL_SHEET = "Уй_хўжалиги_сўровномаси_нарх..."    # sheet name in original file if needed, else leave ""

# ── COLUMN NAMES ───────────────────────────────────────────────────────────

ORIGINAL_ID_COL  = "Уй хўжаликларининг IDси: P999M999999T0HHID0000000000"
FLAGGED_ID_COL   = "m05uuid"

EXTRA_COLUMNS = [
    "Уй хўжаликларининг IDси: P999M999999T0HHID0000000000",
    "1 Интервьюер коди",
    "province_label",
    "mahalla_label",
    "5 Респондентнинг исми ва шарифи:"
]

CITY_COLUMN = "province_label"

# ── LOAD FILES ─────────────────────────────────────────────────────────────

flagged_df = pd.read_excel(r"C:\Users\s.bektemirov\Desktop\Python_VS\all_colored_errors_c.xlsx")
print(f"Flagged rows loaded:   {len(flagged_df)}")

if ORIGINAL_SHEET:
    original_df = pd.read_excel(ORIGINAL_PATH, sheet_name=ORIGINAL_SHEET)
else:
    original_df = pd.read_excel(ORIGINAL_PATH)
print(f"Original rows loaded:  {len(original_df)}")

# ── JOIN EXTRA COLUMNS FROM ORIGINAL ───────────────────────────────────────

original_subset = original_df[EXTRA_COLUMNS].drop_duplicates(subset=[ORIGINAL_ID_COL])

merged_df = flagged_df.merge(
    original_subset,
    left_on  = FLAGGED_ID_COL,
    right_on = ORIGINAL_ID_COL,
    how      = 'left'
)

print(f"Merged rows:           {len(merged_df)}")
print(f"Unmatched rows:        {merged_df[CITY_COLUMN].isna().sum()}")

# ── REORDER COLUMNS — ID + EXTRA COLS FIRST ────────────────────────────────

extra_cols_in_df = [col for col in EXTRA_COLUMNS if col in merged_df.columns]
other_cols       = [col for col in merged_df.columns if col not in extra_cols_in_df]
merged_df        = merged_df[extra_cols_in_df + other_cols]

# ── SPLIT BY CITY AND SAVE ─────────────────────────────────────────────────

import os

cities = [
    "Qoraqalpog'iston Respublikasi",
    "Farg'ona viloyati",
    "Andijon viloyati",
    "Xorazm viloyati",
    "Namangan viloyati",
    "Buxoro viloyati",
    "Qashqadaryo viloyati",
    "Surxondaryo viloyati",
    "Navoiy viloyati",
    "Toshkent viloyati",
    "Samarqand viloyati",
    "Toshkent shahri",
    "Sirdaryo viloyati",
    "Jizzax viloyati"
]

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

for city in cities:
    city_df = merged_df[merged_df[CITY_COLUMN] == city]

    if len(city_df) == 0:
        print(f"  ⚠ No errors found for: {city} — skipping")
        continue

    # use city name as filename, replace special characters
    safe_name = city.replace("'", "").replace(" ", "_")
    out_path  = os.path.join(OUTPUT_FOLDER, f"{safe_name}_errors.xlsx")

    city_df.to_excel(out_path, index=False)
    print(f"  ✓ {city}: {len(city_df)} rows → {safe_name}_errors.xlsx")

print(f"\nDone. City files saved to: {OUTPUT_FOLDER}")

Flagged rows loaded:   25
Original rows loaded:  1213
Merged rows:           25
Unmatched rows:        0
  ✓ Qoraqalpog'iston Respublikasi: 4 rows → Qoraqalpogiston_Respublikasi_errors.xlsx
  ✓ Farg'ona viloyati: 11 rows → Fargona_viloyati_errors.xlsx
  ⚠ No errors found for: Andijon viloyati — skipping
  ⚠ No errors found for: Xorazm viloyati — skipping
  ✓ Namangan viloyati: 1 rows → Namangan_viloyati_errors.xlsx
  ✓ Buxoro viloyati: 1 rows → Buxoro_viloyati_errors.xlsx
  ⚠ No errors found for: Qashqadaryo viloyati — skipping
  ⚠ No errors found for: Surxondaryo viloyati — skipping
  ✓ Navoiy viloyati: 2 rows → Navoiy_viloyati_errors.xlsx
  ⚠ No errors found for: Toshkent viloyati — skipping
  ⚠ No errors found for: Samarqand viloyati — skipping
  ⚠ No errors found for: Toshkent shahri — skipping
  ✓ Sirdaryo viloyati: 4 rows → Sirdaryo_viloyati_errors.xlsx
  ✓ Jizzax viloyati: 2 rows → Jizzax_viloyati_errors.xlsx

Done. City files saved to: C:\Users\s.bektemirov\Desktop\Python_VS\fo